In [1]:
# /home/mitani/vad-model_02.ipynbの入力をOGVCからCSJに変更
# CSJ音声から感情次元表現（Valence, Arousal, Dominance）を抽出してCSV保存 (CSJ_VAD_results.csv)

import os
import csv
from pathlib import Path

import numpy as np
import soundfile as sf
import librosa
import torch
import torch.nn as nn

from transformers import Wav2Vec2Processor
from transformers.models.wav2vec2.modeling_wav2vec2 import (
    Wav2Vec2Model,
    Wav2Vec2PreTrainedModel,
)

from tqdm import tqdm

# ===== 設定 =====
ROOT_PATH = "/home/mitani/CSJ-emo-int_bunseki/ALL-WAV"
OUTPUT_DIR = "CSJ_VAD_results"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAMPLE_RATE = 16000

MODEL_NAME = "audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim"


# ===== モデル定義 =====
class RegressionHead(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.dropout = nn.Dropout(config.final_dropout)
        self.out_proj = nn.Linear(config.hidden_size, config.num_labels)

    def forward(self, features, **kwargs):
        x = features

        x = self.dropout(x)
        x = self.dense(x)
        x = torch.tanh(x)
        x = self.dropout(x)

        x = self.out_proj(x)

        return x


class EmotionModel(Wav2Vec2PreTrainedModel):
    def __init__(self, config):
        super().__init__(config)

        self.config = config
        self.wav2vec2 = Wav2Vec2Model(config)
        self.classifier = RegressionHead(config)

        self.init_weights()

    def forward(self, input_values):
        outputs = self.wav2vec2(input_values)

        hidden_states = outputs[0]

        # 時間方向平均
        hidden_states = torch.mean(hidden_states, dim=1)

        logits = self.classifier(hidden_states)

        return hidden_states, logits


# ===== モデルロード =====
processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)

model = EmotionModel.from_pretrained(MODEL_NAME).to(DEVICE)

model.eval()


# ===== WAV収集 =====
def collect_wav_files(root_dir):
    return sorted(Path(root_dir).glob("*.wav"))


# ===== 音声読み込み =====
def load_audio(path, target_sr=SAMPLE_RATE):

    wav, sr = sf.read(path)

    # ステレオ → モノラル
    if wav.ndim > 1:
        wav = wav.mean(axis=1)

    # リサンプリング
    if sr != target_sr:
        wav = librosa.resample(
            y=wav,
            orig_sr=sr,
            target_sr=target_sr
        )

    return wav.astype(np.float32), target_sr


# ===== 推論 =====
def extract_vad(wav, sampling_rate):

    inputs = processor(
        wav,
        sampling_rate=sampling_rate,
        return_tensors="pt"
    )

    input_values = inputs["input_values"].to(DEVICE)

    with torch.no_grad():
        logits = model(input_values)[1]

    logits = logits.detach().cpu().numpy()[0]

    # モデル出力順:
    # [arousal, dominance, valence]

    arousal = float(logits[0])
    dominance = float(logits[1])
    valence = float(logits[2])

    return valence, arousal, dominance


# ===== メイン処理 =====
def process_csj():

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    wav_files = collect_wav_files(ROOT_PATH)

    results = []

    for wav_path in tqdm(wav_files, desc="Processing CSJ WAV"):

        try:
            wav, sr = load_audio(str(wav_path))

            valence, arousal, dominance = extract_vad(wav, sr)

            results.append([
                wav_path.name,
                valence,
                arousal,
                dominance
            ])

        except Exception as e:
            print(f"Error processing {wav_path.name}: {e}")

    # ===== CSV保存 =====
    csv_path = os.path.join(
        OUTPUT_DIR,
        "CSJ_VAD_results.csv"
    )

    with open(csv_path, "w", newline="", encoding="utf-8") as f:

        writer = csv.writer(f)

        writer.writerow([
            "audio_file",
            "valence",
            "arousal",
            "dominance"
        ])

        writer.writerows(results)

    print(f"\nSaved results to:")
    print(csv_path)


# ===== 実行 =====
if __name__ == "__main__":
    process_csj()

/home/mitani/.conda/envs/ser/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Processing CSJ WAV: 100%|██████████| 522/522 [00:07<00:00, 69.27it/s]


Saved results to:
CSJ_VAD_results/CSJ_VAD_results.csv
